# 58 - Undersampling + Conf60 (Kombinasi 2 Strategi Terbaik)

**Motivasi:** Menggabungkan 2 strategi yang terbukti efektif:
1. **Confidence Filtering >= 60%** (best F1: 0.521, +26%)
2. **Undersampling Neutral** (sad F1: 0.061 -> 0.581)

**Scope:** Fokus ke **Under-660 saja** (sweet spot dari eksperimen sebelumnya).
- Under-382 dan Under-114 tidak dijalankan karena hasil sebelumnya menurun signifikan.

**Perbandingan:**

| Variasi | Neutral Train | Total Train |
|---------|:------------:|:-----------:|
| Conf60 Original | ~5,691 | ~6,795 |
| Conf60 Under-660 | 660 | ~1,700 |

**Model:** Top 3 (Intermediate TL, Late Fusion, FCNN) -- 4-class B1

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusionTransfer,
)
from training.utils import (
    EmotionImageDataset, EmotionLandmarkDataset, EmotionMultimodalDataset,
    train_model, full_evaluation,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR = PROJECT_ROOT / "models" / "frontonly_conf60" / "undersampled"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
NUM_CLASSES = 4
EMOTIONS = ["neutral", "happy", "sad", "negative"]

# Fokus: conf60 original vs conf60 under_660 only
DATASETS = {
    "conf60_original": PROJECT_ROOT / "data" / "dataset_frontonly_conf60_4class",
    "conf60_under_660": PROJECT_ROOT / "data" / "dataset_frontonly_conf60_under_660_4class",
}

for name, path in DATASETS.items():
    if path.exists():
        y = np.load(path / "y_train.npy")
        counts = Counter(y.tolist())
        print(f"  {name}: {len(y)} train | {dict(sorted(counts.items()))}")
    else:
        print(f"  {name}: NOT FOUND")

print("\nSetup complete.")

Device: cuda
GPU: Tesla T4
  conf60_original: 5287 train | {0: 4526, 1: 416, 2: 287, 3: 58}
  conf60_under_660: 1421 train | {0: 660, 1: 416, 2: 287, 3: 58}

Setup complete.


## Helper Functions

In [2]:
def load_loaders(dataset_dir, model_type, batch_size=32):
    if model_type == "cnn":
        tr = EmotionImageDataset(dataset_dir/"X_train_images.npy", dataset_dir/"y_train.npy")
        vl = EmotionImageDataset(dataset_dir/"X_val_images.npy", dataset_dir/"y_val.npy")
        te = EmotionImageDataset(dataset_dir/"X_test_images.npy", dataset_dir/"y_test.npy")
    elif model_type == "fcnn":
        tr = EmotionLandmarkDataset(dataset_dir/"X_train_landmarks.npy", dataset_dir/"y_train.npy")
        vl = EmotionLandmarkDataset(dataset_dir/"X_val_landmarks.npy", dataset_dir/"y_val.npy")
        te = EmotionLandmarkDataset(dataset_dir/"X_test_landmarks.npy", dataset_dir/"y_test.npy")
    else:
        tr = EmotionMultimodalDataset(dataset_dir/"X_train_images.npy", dataset_dir/"X_train_landmarks.npy", dataset_dir/"y_train.npy")
        vl = EmotionMultimodalDataset(dataset_dir/"X_val_images.npy", dataset_dir/"X_val_landmarks.npy", dataset_dir/"y_val.npy")
        te = EmotionMultimodalDataset(dataset_dir/"X_test_images.npy", dataset_dir/"X_test_landmarks.npy", dataset_dir/"y_test.npy")
    return (DataLoader(tr, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True),
            DataLoader(vl, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
            DataLoader(te, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True))


def run_experiment(name, ModelClass, model_type, lr, dataset_dir):
    tr, vl, te = load_loaders(dataset_dir, model_type, BATCH_SIZE)
    model = ModelClass(num_classes=NUM_CLASSES).to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    sp = str(OUTPUT_DIR / f"{name}.pth")
    print(f"\n>> {name}")
    train_model(model, tr, vl, crit, opt, sch, device, model_type, EPOCHS, PATIENCE, sp)
    model.load_state_dict(torch.load(sp, map_location=device, weights_only=True))
    r = full_evaluation(model, te, crit, device, model_type, EMOTIONS)
    print(f"   Acc={r['test_accuracy']:.4f} Macro-F1={r['test_macro_f1']:.4f}")

    # Per-class
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in te:
            if model_type == "fusion":
                imgs, lms, lbls = batch
                out = model(imgs.to(device), lms.to(device))
            elif model_type == "cnn":
                imgs, lbls = batch
                out = model(imgs.to(device))
            else:
                lms, lbls = batch
                out = model(lms.to(device))
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(lbls.numpy())
    print(classification_report(all_labels, all_preds, target_names=EMOTIONS, zero_division=0))
    return {
        "accuracy": float(r["test_accuracy"]),
        "macro_f1": float(r["test_macro_f1"]),
        "weighted_f1": float(r["test_weighted_f1"]),
        "per_class": classification_report(all_labels, all_preds, target_names=EMOTIONS, output_dict=True, zero_division=0),
    }


def late_fusion_exp(name, dataset_dir):
    tr_img, vl_img, _ = load_loaders(dataset_dir, "cnn", BATCH_SIZE)
    tr_lm, vl_lm, _ = load_loaders(dataset_dir, "fcnn", BATCH_SIZE)
    _, _, te_mm = load_loaders(dataset_dir, "fusion", BATCH_SIZE)

    cnn = EmotionCNN(num_classes=NUM_CLASSES).to(device)
    o1 = torch.optim.Adam(cnn.parameters(), lr=0.0001)
    s1 = torch.optim.lr_scheduler.ReduceLROnPlateau(o1, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(cnn, tr_img, vl_img, nn.CrossEntropyLoss(), o1, s1, device, "cnn", EPOCHS, PATIENCE, str(OUTPUT_DIR/f"{name}_cnn.pth"))

    fcnn = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
    o2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
    s2 = torch.optim.lr_scheduler.ReduceLROnPlateau(o2, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(fcnn, tr_lm, vl_lm, nn.CrossEntropyLoss(), o2, s2, device, "fcnn", EPOCHS, PATIENCE, str(OUTPUT_DIR/f"{name}_fcnn.pth"))

    cnn.load_state_dict(torch.load(OUTPUT_DIR/f"{name}_cnn.pth", map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(OUTPUT_DIR/f"{name}_fcnn.pth", map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()

    cp, fp, lb = [], [], []
    with torch.no_grad():
        for imgs, lms, lbls in te_mm:
            cp.append(torch.softmax(cnn(imgs.to(device)), dim=1).cpu().numpy())
            fp.append(torch.softmax(fcnn(lms.to(device)), dim=1).cpu().numpy())
            lb.append(lbls.numpy())
    cp = np.concatenate(cp); fp = np.concatenate(fp); lb = np.concatenate(lb)

    best_f1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        preds = (w*cp+(1-w)*fp).argmax(1)
        f1 = f1_score(lb, preds, average="macro", zero_division=0)
        if f1 > best_f1: best_f1=f1; best_w=w; best_preds=preds

    acc = accuracy_score(lb, best_preds)
    wf1 = f1_score(lb, best_preds, average="weighted", zero_division=0)
    print(f"\n>> {name} (w={best_w:.2f}): Acc={acc:.4f} F1={best_f1:.4f}")
    print(classification_report(lb, best_preds, target_names=EMOTIONS, zero_division=0))
    return {"accuracy": acc, "macro_f1": best_f1, "weighted_f1": wf1, "best_cnn_weight": best_w,
            "per_class": classification_report(lb, best_preds, target_names=EMOTIONS, output_dict=True, zero_division=0)}

print("Helper ready.")

Helper ready.


## Run Experiments

In [3]:
all_results = {}

for ds_name, ds_path in DATASETS.items():
    if not ds_path.exists():
        print(f"SKIP: {ds_name} not found")
        continue
    print(f"\n{'='*70}\n  DATASET: {ds_name}\n{'='*70}")

    r = run_experiment(f"IntTL_{ds_name}", IntermediateFusionTransfer, "fusion", 0.00005, ds_path)
    all_results[f"IntTL_{ds_name}"] = r

    r = run_experiment(f"FCNN_{ds_name}", EmotionFCNN, "fcnn", 0.0001, ds_path)
    all_results[f"FCNN_{ds_name}"] = r

    r = late_fusion_exp(f"LateFusion_{ds_name}", ds_path)
    all_results[f"LateFusion_{ds_name}"] = r


  DATASET: conf60_original



>> IntTL_conf60_original
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.1815     0.5046     1.0740    0.8048   0.2512   0.000050  (19.3s)


     2      0.6790     0.8320     0.8510    0.8048   0.3337   0.000050  (19.2s)


     3      0.4800     0.8744     0.7373    0.8359   0.4190   0.000050  (19.1s)


...

    42      0.7011     0.7431     0.6310    0.8169   0.3985   0.000050  (0.8s)


    43      0.7036     0.7340     0.6111    0.8307   0.4062   0.000050  (0.9s)


    44      0.6848     0.7579     0.6326    0.8117   0.4345   0.000050  (0.8s)


    45      0.6977     0.7410     0.5938    0.8238   0.4229   0.000050  (0.8s)

Early stopping at epoch 45. Best epoch: 30 (val_f1=0.4525)

Best: epoch 30, val_acc=0.8428, val_f1=0.4525
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/undersampled/LateFusion_conf60_under_660_fcnn.pth



>> LateFusion_conf60_under_660 (w=0.25): Acc=0.8321 F1=0.5080
              precision    recall  f1-score   support

     neutral       0.88      0.91      0.89       688
       happy       0.76      0.67      0.71       183
         sad       0.43      0.42      0.42        50
    negative       0.00      0.00      0.00         8

    accuracy                           0.83       929
   macro avg       0.52      0.50      0.51       929
weighted avg       0.82      0.83      0.83       929



## Ringkasan

In [4]:
print("="*85)
print("RINGKASAN: UNDERSAMPLING + CONF60")
print("="*85)

models = ["IntTL", "FCNN", "LateFusion"]
ds_list = ["conf60_original", "conf60_under_660"]

for m in models:
    print(f"\n  {m}:")
    print(f"  {'Dataset':<22} {'Macro F1':>10} {'Acc':>10} {'neutral':>10} {'happy':>10} {'sad':>10} {'negative':>10}")
    print(f"  {'-'*82}")
    for ds in ds_list:
        k = f"{m}_{ds}"
        if k in all_results:
            r = all_results[k]
            pc = r.get("per_class", {})
            print(f"  {ds:<22} {r['macro_f1']:>10.4f} {r['accuracy']:>10.4f}"
                  f" {pc.get('neutral',{}).get('f1-score',0):>10.3f}"
                  f" {pc.get('happy',{}).get('f1-score',0):>10.3f}"
                  f" {pc.get('sad',{}).get('f1-score',0):>10.3f}"
                  f" {pc.get('negative',{}).get('f1-score',0):>10.3f}")

with open(OUTPUT_DIR / "undersampled_conf60_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=float)
print(f"\nSaved: {OUTPUT_DIR / 'undersampled_conf60_results.json'}")

RINGKASAN: UNDERSAMPLING + CONF60

  IntTL:
  Dataset                  Macro F1        Acc    neutral      happy        sad   negative
  ----------------------------------------------------------------------------------
  conf60_original            0.4784     0.7933      0.875      0.628      0.410      0.000
  conf60_under_660           0.4013     0.5694      0.635      0.497      0.473      0.000

  FCNN:
  Dataset                  Macro F1        Acc    neutral      happy        sad   negative
  ----------------------------------------------------------------------------------
  conf60_original            0.4886     0.8202      0.887      0.715      0.353      0.000
  conf60_under_660           0.4334     0.8062      0.878      0.713      0.143      0.000

  LateFusion:
  Dataset                  Macro F1        Acc    neutral      happy        sad   negative
  ----------------------------------------------------------------------------------
  conf60_original            0.4655     